# RLDS 数据格式解析

> 目标：用本地 `droid_100` 样例搞清 **RLDS / TFDS** 的 Episode→Steps 结构、observation/action、以及「action 语义要看 description」。  
> 样例：`samples/tensorflow_datasets/droid_100/1.0.0`（约 2GB，若无请 `./scripts/download_samples.sh --all`）。

**依赖**：
```bash
pip install -r ../requirements-rlds.txt
# 或: pip install tensorflow tensorflow-datasets
```

无本地数据时用官方 Colab：  
https://colab.research.google.com/github/google-deepmind/open_x_embodiment/blob/main/colabs/Open_X_Embodiment_Datasets.ipynb


## 1. 一句话：RLDS 是什么

**RLDS** = Reinforcement Learning Datasets（Google/DeepMind）：用统一的 **Episode / Step** 逻辑 schema 存轨迹。  
磁盘通常是 **TFRecord**，经 **TensorFlow Datasets (TFDS)** 加载。

```text
Dataset (tf.data)
 └── Episode
      ├── episode_metadata   # 文件路径、机器人相关元信息…
      └── steps              # 嵌套 Dataset，长度 = T
           ├── observation   # 图像、state、…
           ├── action        # float 向量（语义因数据集而异！）
           ├── language_instruction
           ├── reward / discount / is_first / is_last / is_terminal
```

**名字带 RL，但模仿学习/VLA 一样用**——很多集的 reward 为 0，真正标签是 `action`。


In [ ]:
from pathlib import Path
import json

REPO = Path("..").resolve()
DATA_DIR = REPO / "samples" / "tensorflow_datasets" / "droid_100" / "1.0.0"
print("Expect:", DATA_DIR)
print("exists:", DATA_DIR.exists())
if DATA_DIR.exists():
    print("files:", sorted(p.name for p in DATA_DIR.iterdir())[:8], "...")
else:
    print("请先下载 droid_100，或改 DATA_DIR 指向你的 TFDS version 目录")


## 2. 先读元数据：`dataset_info.json` + `features.json`

不先 `as_dataset` 也能知道字段和 **action 官方描述**（这比 shape 更重要）。


In [ ]:
assert DATA_DIR.exists(), "缺少 droid_100 样例"
info = json.loads((DATA_DIR / "dataset_info.json").read_text(encoding="utf-8"))
features = json.loads((DATA_DIR / "features.json").read_text(encoding="utf-8"))

print("name:", info.get("name"))
print("version:", info.get("version"))
print("fileFormat:", info.get("fileFormat"))
# splits 统计
for split in info.get("splits", [])[:3]:
    print("split:", split.get("name"), "numShards=", split.get("shardLengths") or split.get("shardLengths"))

def walk(node, prefix=""):
    """Recursive print of TFDS features.json descriptions."""
    if not isinstance(node, dict):
        return
    # common wrappers
    if "featuresDict" in node:
        walk(node["featuresDict"], prefix)
        return
    feats = node.get("features", node)
    if not isinstance(feats, dict):
        return
    for k, v in feats.items():
        desc = v.get("description") if isinstance(v, dict) else None
        shape = None
        dtype = None
        if isinstance(v, dict) and "tensor" in v:
            shape = v["tensor"].get("shape", {}).get("dimensions")
            dtype = v["tensor"].get("dtype")
        line = f"{prefix}{k}"
        extra = []
        if dtype: extra.append(f"dtype={dtype}")
        if shape is not None: extra.append(f"shape={shape}")
        if desc: extra.append(f"| {desc}")
        print(line + (("  (" + ", ".join(extra) + ")") if extra else ""))
        if isinstance(v, dict):
            if "featuresDict" in v or (v.get("sequence") and "feature" in v.get("sequence", {})):
                child = v.get("featuresDict") or v["sequence"]["feature"]
                walk(child, prefix + k + ".")
            elif "features" in v:
                walk(v, prefix + k + ".")

print("\n=== Feature tree (with descriptions) ===")
walk(features)


### 关键发现（对本 `droid_100` 样例）

打开上面的 `action` **description**，你会看到它写的是类似：

> Robot action, consists of **[6x joint velocities, 1x gripper position]**

这说明：
1. **RLDS 只规定「有一个叫 action 的 float 向量」**，不规定必须是 delta EEF。
2. 同一数据集往往还有 `action_dict`（cartesian_position / velocity / gripper…）——**更细的语义在字典里**。
3. OpenVLA 论文里说的「DROID absolute EEF」是 **训练时选用/转换后的表示**，不一定等于 TFRecord 里默认的 `action` 字段。

→ **格式对齐 ≠ 直接相信字段名；一定要读 description + 原始论文/代码。**


## 3. 用 TFDS 加载并遍历 Episode / Step


In [ ]:
import numpy as np
import tensorflow_datasets as tfds

builder = tfds.builder_from_directory(str(DATA_DIR))
ds = builder.as_dataset(split="train")
print(builder.info)

# 取 1 条 episode
episode = next(iter(ds.take(1)))
print("\nEpisode top-level keys:", list(episode.keys()))
if "episode_metadata" in episode:
    meta = {k: (v.numpy() if hasattr(v, "numpy") else v) for k, v in episode["episode_metadata"].items()}
    # bytes → str
    for k, v in list(meta.items()):
        if isinstance(v, (bytes, bytearray)):
            meta[k] = v.decode("utf-8", errors="replace")
    print("episode_metadata:", meta)


In [ ]:
steps = episode["steps"]
# steps 是嵌套 tf.data.Dataset
print("steps:", steps)

n = 0
first_step = None
actions = []
for step in steps:
    if first_step is None:
        first_step = step
    actions.append(step["action"].numpy())
    n += 1
actions = np.stack(actions)
print(f"num steps T = {n}")
print(f"action array shape = {actions.shape}  (T, D)")
print(f"action[0] = {np.round(actions[0], 4)}")
print(f"action mean = {np.round(actions.mean(0), 4)}")
print(f"action std  = {np.round(actions.std(0), 4)}")


## 4. 解剖单步：observation / language / flags


In [ ]:
step = first_step
print("Step keys:", list(step.keys()))

# action
a = step["action"].numpy()
print(f"\naction: shape={a.shape}, dtype={a.dtype}")
print(" ", np.round(a, 4))

# language
for key in ["language_instruction", "natural_language_instruction"]:
    if key in step:
        lang = step[key].numpy()
        if isinstance(lang, (bytes, bytearray)):
            lang = lang.decode()
        print(f"{key}: {lang!r}")
    obs = step.get("observation")
    if obs is not None and key in obs:
        lang = obs[key].numpy()
        if isinstance(lang, (bytes, bytearray)):
            lang = lang.decode()
        print(f"observation.{key}: {lang!r}")

# flags
for key in ["is_first", "is_last", "is_terminal", "reward", "discount"]:
    if key in step:
        v = step[key].numpy()
        print(f"{key}: {v}")

# observation 各字段 shape
print("\nobservation keys:")
obs = step["observation"]
for k in sorted(obs.keys()):
    v = obs[k]
    if hasattr(v, "shape") and len(v.shape) > 0:
        print(f"  {k}: shape={tuple(v.shape.as_list() if hasattr(v.shape, 'as_list') else v.shape)}, dtype={v.dtype}")
    else:
        val = v.numpy() if hasattr(v, "numpy") else v
        if isinstance(val, (bytes, bytearray)):
            val = val.decode("utf-8", errors="replace")
        print(f"  {k}: {val}")


In [ ]:
# 可视化一张相机图
import matplotlib.pyplot as plt

obs = first_step["observation"]
img_keys = [k for k in obs.keys() if "image" in k.lower()]
print("image keys:", img_keys)

if img_keys:
    fig, axes = plt.subplots(1, min(3, len(img_keys)), figsize=(4 * min(3, len(img_keys)), 3))
    if min(3, len(img_keys)) == 1:
        axes = [axes]
    for ax, k in zip(axes, img_keys[:3]):
        img = obs[k].numpy()
        ax.imshow(img)
        ax.set_title(k)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("本 step 无 image_* 字段（少见）")


## 5. `action` vs `action_dict`：别混

很多 OXE/DROID 集里：
- `action`：一个「默认」训练向量（本样例是关节速度 + gripper）
- `action_dict.*`：笛卡尔位置/速度、夹爪等，**可供你选**

训练前必须决定：**你的模型要用哪一套**，必要时自己算 delta。


In [ ]:
if "action_dict" in first_step:
    ad = first_step["action_dict"]
    print("action_dict keys:", list(ad.keys()))
    for k in ad.keys():
        v = ad[k].numpy()
        print(f"  {k}: shape={v.shape}, values={np.round(v, 4)}")
else:
    print("无 action_dict（有的子集只有扁平 action）")

# 若有 cartesian_position，演示「绝对位姿 → 简易 delta」
if "action_dict" in first_step and "cartesian_position" in first_step["action_dict"]:
    cartesians = []
    for i, step in enumerate(episode["steps"]):
        cartesians.append(step["action_dict"]["cartesian_position"].numpy())
        if i >= 50:  # 只取前 51 步做 demo
            break
    cartesians = np.stack(cartesians)
    deltas = cartesians[1:] - cartesians[:-1]
    print("\ncartesian_position[0]:", np.round(cartesians[0], 4))
    print("delta[0] (t1-t0):", np.round(deltas[0], 4))
    print("← 这就是 OpenVLA 类「压成 delta」时经常做的事（外加旋转表示转换等）")


## 6. 画动作曲线（默认 `action` 7D）


In [ ]:
T, D = actions.shape
t = np.arange(T)
fig, ax = plt.subplots(figsize=(10, 3.5))
for d in range(D):
    ax.plot(t, actions[:, d], label=f"a[{d}]", alpha=0.8)
ax.set_xlabel("step")
ax.set_ylabel("action")
ax.set_title("Episode 0 · flat `action` over time")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()


## 7. 和 LeRobot 对照

| | RLDS | LeRobot |
|--|------|---------|
| 逻辑单位 | Episode 含嵌套 steps | parquet **一行一帧** |
| 图像 | 常直接在 step.observation 张量里 | 常在独立 mp4 |
| 生态 | TF / JAX / OXE 预训练 | PyTorch / HF 微调 |
| action 语义 | **看 description / action_dict** | 看 `info.json` 的 names/shape |
| 本仓库样例 | `droid_100`（Franka 野外子集） | `aloha_sim`（双臂关节） |

**同一物理机器人数据可以导出成两种格式**；格式只解决「怎么存、怎么读」，不自动统一「动作是 delta 还是 joint」。


## 8. 自测

1. RLDS 的 `steps` 为什么是嵌套 Dataset 而不是固定长度 Tensor？
2. 本 notebook 的默认 `action` 是 delta EEF 吗？依据是哪段 description？
3. 若要用 OpenVLA 风格 7D delta，你该用 `action` 还是从 `action_dict` 自己算？
4. LeRobot 的 `task_index` 对应 RLDS 的哪个字段角色？
